# Baseline — Solve Misconceptions

**Competition:** each student answered several questions; wrong answers reveal
**misconceptions** (e.g. "heavier objects fall faster"). Given a student's answers,
rank the 10 most likely misconception ids from `misconceptions_pool.csv`.

- **Task:** per-student ranking of misconceptions (`misconception_ids` =
  10 space-separated ids)
- **Metric:** rewards placing the student's true misconceptions early
- **Kaggle link:** _TODO: add link_

**Approach (zero-shot text matching):** represent each student by the text of the
questions they answered together with their answers, and each misconception by its
description; rank misconceptions by TF-IDF cosine similarity.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

DATA_DIR = "."
obs = pd.read_csv(f"{DATA_DIR}/observations.csv")
pool = pd.read_csv(f"{DATA_DIR}/misconceptions_pool.csv")
sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
print(obs.shape, pool.shape, sub.shape)
obs.head(3)

(2880, 4) (47, 2) (360, 2)


,student_id,question_id,question,student_answer
0,dev_0000,q016,"Which is larger, 0.1 or 0.09?",0.1
1,dev_0000,q019,A square has side 3. What is its area if you c...,9
2,dev_0000,q025,A 10 kg ball and a 1 kg ball are dropped toget...,The 10 kg ball.


In [2]:
# Student document = their questions + their answers
obs["qa"] = obs["question"].astype(str) + " " + obs["student_answer"].astype(str)
student_docs = obs.groupby("student_id")["qa"].apply(" ".join)

vec = TfidfVectorizer(ngram_range=(1, 2), stop_words="english")
M = vec.fit_transform(pd.concat([student_docs,
                                 pool["misconception_text"]]).values)
S = M[:len(student_docs)]
P = M[len(student_docs):]
sim = cosine_similarity(S, P)          # students x misconceptions
print(sim.shape)

(360, 47)


In [3]:
order = np.argsort(-sim, axis=1)[:, :10]
ids = pool["misconception_id"].values
ranked = [" ".join(ids[row]) for row in order]
out = pd.DataFrame({"student_id": student_docs.index, "misconception_ids": ranked})
out = sub[["student_id"]].merge(out, on="student_id", how="left")
out.to_csv("submission.csv", index=False)
out.head()

,student_id,misconception_ids
0,dev_0000,m033 m036 m032 m034 m014 m017 m040 m021 m018 m001
1,dev_0001,m020 m034 m013 m023 m004 m021 m002 m018 m042 m015
2,dev_0002,m020 m033 m010 m013 m040 m042 m002 m001 m023 m024
3,dev_0003,m029 m027 m017 m040 m025 m042 m037 m031 m028 m045
4,dev_0004,m010 m024 m036 m011 m022 m033 m035 m002 m013 m017


## Ideas to improve

- Only *wrong* answers carry misconception signal — detect correct answers (e.g. ask
  an LLM or use heuristics) and exclude them from the student document.
- Use **sentence embeddings** (MiniLM) instead of TF-IDF for semantic matching:
  "The 10 kg ball hits first" should match the gravity misconception even with no
  shared words.
- Best approach: ask an **LLM** per (question, wrong answer) pair which misconception
  from the pool explains it, then aggregate per student.
